<a href="https://colab.research.google.com/github/Minayaterry/lab16/blob/develop/Lab16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEMANA 16: DESARROLLO DE APLICACIONES DE MACHINE LEARNING

### Nombres:
**Terry Minaya Torres                                            
Ronald Tuncard Andia**

# 1. Preprocesamiento de la información

### a. Carga y unión de los datos

In [1]:
import pandas as pd

# Cargar los datos
url_data = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
url_test = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

# Definir los nombres de las columnas
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]

# Leer los datos
df_train = pd.read_csv(url_data, names=columns, sep=",\s*", engine="python")
df_test = pd.read_csv(url_test, names=columns, sep=",\s*", engine="python", skiprows=1)

# Unir los datos
df = pd.concat([df_train, df_test], ignore_index=True)

### b. Corrección de errores y datos faltantes

In [2]:
# Reemplazar '?' por NaN
df.replace("?", pd.NA, inplace=True)

# Verificar valores faltantes
print(df.isnull().sum())

# Imputar datos faltantes (ejemplo: moda para categóricas, mediana para numéricas)
for col in df.columns:
    if df[col].dtype == "object":
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
income               0
dtype: int64


/tmp/ipython-input-2-1649892374.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipython-input-2-1649892374.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

### c. Tratamiento de outliers


In [3]:
import numpy as np

# Función para detectar outliers usando IQR
def detect_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (series < lower_bound) | (series > upper_bound)

# Aplicar a variables numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    outliers = detect_outliers(df[col])
    df[col] = np.where(outliers, df[col].median(), df[col])

### d. Transformación de variables y análisis de correlación

In [4]:
# Convertir income a binario (0 para <=50K, 1 para >50K)
df["income"] = df["income"].apply(lambda x: 1 if ">50K" in x else 0)

# Análisis de correlaciónS
correlation_matrix = df[numeric_cols].corr()
print(correlation_matrix)

                     age    fnlwgt  education-num  capital-gain  capital-loss  \
age             1.000000 -0.065548       0.104938           NaN           NaN   
fnlwgt         -0.065548  1.000000      -0.015439           NaN           NaN   
education-num   0.104938 -0.015439       1.000000           NaN           NaN   
capital-gain         NaN       NaN            NaN           NaN           NaN   
capital-loss         NaN       NaN            NaN           NaN           NaN   
hours-per-week  0.048429 -0.004181       0.125656           NaN           NaN   

                hours-per-week  
age                   0.048429  
fnlwgt               -0.004181  
education-num         0.125656  
capital-gain               NaN  
capital-loss               NaN  
hours-per-week        1.000000  


# 2. Selección de variables y preparación de datos

### a. Cálculo del Information Value (IV)

In [5]:
from sklearn.feature_selection import mutual_info_classif

# Separar características y objetivo
X = df.drop("income", axis=1)
y = df["income"]

# Convertir categóricas a numéricas para el cálculo de IV
X_encoded = pd.get_dummies(X, drop_first=True)

# Calcular IV
iv = mutual_info_classif(X_encoded, y)
iv_series = pd.Series(iv, index=X_encoded.columns)
print(iv_series.sort_values(ascending=False))

marital-status_Married-civ-spouse    0.109648
age                                  0.069988
marital-status_Never-married         0.065446
education-num                        0.062370
relationship_Own-child               0.039882
                                       ...   
native-country_Poland                0.000000
native-country_Thailand              0.000000
native-country_Taiwan                0.000000
native-country_Trinadad&Tobago       0.000000
native-country_Yugoslavia            0.000000
Length: 97, dtype: float64


### b. Escalado y conversión a dummies

In [6]:
from sklearn.preprocessing import StandardScaler

# Seleccionar variables numéricas y categóricas
numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# Escalar numéricas
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# Convertir categóricas a dummies
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

### c. División y balanceo de clases

In [7]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Balancear clases con SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# 3. Entrenamiento y evaluación de modelos

### a. Entrenamiento con hiperparámetros estándar

In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Modelos
models = {
    "k-NN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

# Entrenar y evaluar
for name, model in models.items():
    model.fit(X_train_balanced, y_train_balanced)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name}: Accuracy = {accuracy:.4f}")

k-NN: Accuracy = 0.7734
SVM: Accuracy = 0.7897


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: Accuracy = 0.7802
Decision Tree: Accuracy = 0.7658
Random Forest: Accuracy = 0.8124


### b. Búsqueda en cuadrícula y aleatoria

In [9]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Hiperparámetros para Random Forest
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

# Búsqueda en cuadrícula
grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
grid_search.fit(X_train_balanced, y_train_balanced)
print(f"Mejor accuracy (Grid Search): {grid_search.best_score_:.4f}")
print(f"Mejores parámetros: {grid_search.best_params_}")

# Búsqueda aleatoria
random_search = RandomizedSearchCV(RandomForestClassifier(), param_grid, n_iter=10, cv=5)
random_search.fit(X_train_balanced, y_train_balanced)
print(f"Mejor accuracy (Random Search): {random_search.best_score_:.4f}")
print(f"Mejores parámetros: {random_search.best_params_}")

Mejor accuracy (Grid Search): 0.8787
Mejores parámetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Mejor accuracy (Random Search): 0.8790
Mejores parámetros: {'n_estimators': 300, 'min_samples_split': 2, 'max_depth': None}


### c. Mejor modelo y resultados

In [10]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Mejor modelo: {best_model}")
print(f"Accuracy en prueba: {accuracy:.4f}")

Mejor modelo: RandomForestClassifier()
Accuracy en prueba: 0.8109
